# Vocal-Guard — Baseline B0: LFCC-GMM pada ASVspoof 2019 LA
**Penelitian:** Spoofing-Aware Audio Front-End yang Efisien untuk Robot Sosial  
**Fase 0:** Target 13 Oktober 2026 — Baseline Sanity Check & Single Source of Truth  

---

### Tujuan Notebook Ini:
1. Menyiapkan dataset resmi **ASVspoof 2019 LA** (download langsung dari Zenodo mirror jika belum ada).
2. Menjalankan pipeline ekstraksi fitur **LFCC** (20 koefisien + delta + delta-delta = 60 dimensi) sebagai *Single Source of Truth*.
3. Melatih baseline klasik **B0: LFCC-GMM** (Gaussian Mixture Model untuk *bonafide* vs *spoof*).
4. Menghitung **Equal Error Rate (EER)** resmi pada *development set* sebagai patokan evaluasi (Gerbang Keputusan G1).

In [ ]:
# 1. Setup Dependencies
!pip install -q soundfile librosa scikit-learn pandas scipy tqdm matplotlib

In [ ]:
import os
import sys
import glob
import zipfile
import pickle
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import scipy.signal
from scipy.fftpack import dct
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print(f"NumPy version: {np.__version__}")

## 2. Persiapan Data ASVspoof 2019 LA
Mengunduh LA.zip (7.6 GB) dari Zenodo mirror resmi dan mengekstrak berkas protokol serta audio.


In [ ]:
import os
from pathlib import Path

# Jalur dataset spesifik yang terpasang di Kaggle
KAGGLE_SPECIFIC_ROOT = Path('/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA')
IS_KAGGLE = os.path.exists('/kaggle')
BASE_DIR = Path('/kaggle/working' if IS_KAGGLE else './')

if KAGGLE_SPECIFIC_ROOT.exists():
    LA_ROOT = KAGGLE_SPECIFIC_ROOT
    PROTOCOL_DIR = LA_ROOT / 'ASVspoof2019_LA_cm_protocols'
    print(f'✅ Dataset ASVspoof 2019 LA terdeteksi di: {LA_ROOT}')
else:
    # Pencarian fleksibel alternatif
    kaggle_protocols = list(Path('/kaggle/input').glob('**/ASVspoof2019.LA.cm.train.trn.txt')) if IS_KAGGLE else []
    if kaggle_protocols:
        PROTOCOL_DIR = kaggle_protocols[0].parent
        LA_ROOT = PROTOCOL_DIR.parent
        print(f'✅ Dataset ditemukan di: {LA_ROOT}')
    else:
        raise FileNotFoundError('Folder dataset LA tidak ditemukan di /kaggle/input!')

print(f'LA Root:      {LA_ROOT}')
print(f'Protocol Dir: {PROTOCOL_DIR}')


## 3. Ekstraksi Fitur LFCC (*Single Source of Truth*)
Spesifikasi sesuai proposal:
- **Pre-emphasis:** $\alpha = 0.97$
- **Windowing:** Hann window 20 ms, hop 10 ms
- **FFT:** 512-point FFT
- **Filterbank:** 20 filter segitiga linier (linear triangular filterbank)
- **Log Compression:** $\ln(\text{energy} + 10^{-12})$
- **DCT:** Tipe 2 ortho $\rightarrow$ 20 koefisien cepstral
- **Deltas:** Statis (20) + Delta (20) + Delta-Delta (20) = **60 dimensi** per frame

In [ ]:
def get_linear_filterbank(n_filters=20, n_fft=512, sr=16000, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sr / 2.0
    freq_points = np.linspace(low_freq, high_freq, n_filters + 2)
    bin_points = np.floor((n_fft + 1) * freq_points / sr).astype(int)
    n_bins = n_fft // 2 + 1
    filterbank = np.zeros((n_filters, n_bins))
    for i in range(n_filters):
        left, center, right = bin_points[i], bin_points[i + 1], bin_points[i + 2]
        if center > left:
            for b in range(left, center):
                if b < n_bins:
                    filterbank[i, b] = (b - left) / (center - left)
        if right > center:
            for b in range(center, right):
                if b < n_bins:
                    filterbank[i, b] = (right - b) / (right - center)
    return filterbank

def compute_deltas(feats, N=2):
    T, D = feats.shape
    deltas = np.zeros_like(feats)
    denom = 2 * sum(n**2 for n in range(1, N + 1))
    for t in range(T):
        num = np.zeros(D)
        for n in range(1, N + 1):
            t_plus = min(T - 1, t + n)
            t_minus = max(0, t - n)
            num += n * (feats[t_plus] - feats[t_minus])
        deltas[t] = num / denom
    return deltas

def extract_lfcc_features(y, sr=16000, window_ms=20.0, hop_ms=10.0, n_fft=512, num_filters=20, num_coeffs=20):
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    # 1. Pre-emphasis
    y = np.append(y[0], y[1:] - 0.97 * y[:-1])
    
    frame_len = int(sr * window_ms / 1000.0)  # 320
    hop_len = int(sr * hop_ms / 1000.0)      # 160
    if len(y) < frame_len:
        y = np.pad(y, (0, frame_len - len(y)))
    
    num_frames = 1 + (len(y) - frame_len) // hop_len
    shape = (num_frames, frame_len)
    strides = (y.strides[0] * hop_len, y.strides[0])
    frames = np.lib.stride_tricks.as_strided(y, shape=shape, strides=strides).copy()
    frames *= np.hanning(frame_len)
    
    mag_spec = np.abs(np.fft.rfft(frames, n=n_fft, axis=-1))
    fbank = get_linear_filterbank(n_filters=num_filters, n_fft=n_fft, sr=sr)
    filter_energies = np.dot(mag_spec, fbank.T)
    log_energies = np.log(filter_energies + 1e-12)
    
    static_lfcc = dct(log_energies, type=2, axis=-1, norm='ortho')[:, :num_coeffs]
    delta = compute_deltas(static_lfcc, N=2)
    delta2 = compute_deltas(delta, N=2)
    
    return np.hstack([static_lfcc, delta, delta2])  # (T, 60)

## 4. Pembacaan Protokol Resmi ASVspoof 2019 LA

In [ ]:
def load_protocol(protocol_file):
    df = pd.read_csv(
        protocol_file,
        sep=r'\s+',
        header=None,
        names=['speaker_id', 'audio_id', 'system_id', 'attack_id', 'label'],
        dtype=str
    )
    df['label'] = df['label'].str.lower()
    return df

train_protocol_file = list(PROTOCOL_DIR.glob('*train.trn.txt'))[0]
dev_protocol_file = list(PROTOCOL_DIR.glob('*dev.trl.txt'))[0]

train_df = load_protocol(train_protocol_file)
dev_df = load_protocol(dev_protocol_file)

print(f"Train samples: {len(train_df)} (Bonafide: {(train_df['label'] == 'bonafide').sum()}, Spoof: {(train_df['label'] == 'spoof').sum()})")
print(f"Dev samples:   {len(dev_df)} (Bonafide: {(dev_df['label'] == 'bonafide').sum()}, Spoof: {(dev_df['label'] == 'spoof').sum()})")

# Temukan folder FLAC secara fleksibel
def find_flac_dir(root_dir, split_name):
    candidates = list(root_dir.glob(f'**/*{split_name}*/**/flac'))
    if not candidates:
        candidates = list(root_dir.glob(f'**/*{split_name}*/**/FLAC'))
    if not candidates:
        candidates = list(Path('/kaggle/input').glob(f'**/*{split_name}*/**/flac'))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"Folder FLAC untuk split '{split_name}' tidak ditemukan di {root_dir}")

train_audio_dir = find_flac_dir(LA_ROOT, 'train')
dev_audio_dir = find_flac_dir(LA_ROOT, 'dev')

print(f"Train audio dir: {train_audio_dir}")
print(f"Dev audio dir:   {dev_audio_dir}")


## 5. Ekstraksi Fitur untuk Training Baseline B0
Kita mengumpulkan frame LFCC untuk kelas `bonafide` dan `spoof`.  
*Catatan:* Untuk menjaga waktu training efisien dan representatif, kita dapat melatih GMM dengan subset acak frame dari seluruh utterance.

In [ ]:
def collect_training_frames(df, audio_dir, target_label, max_utterances=None, max_frames=500000):
    subset = df[df['label'] == target_label]
    if max_utterances and len(subset) > max_utterances:
        subset = subset.sample(n=max_utterances, random_state=42)
    
    all_frames = []
    print(f"Mengumpulkan frame untuk {target_label.upper()} ({len(subset)} utterance)... status:")
    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        file_path = audio_dir / f"{row['audio_id']}.flac"
        if not file_path.exists():
            continue
        y, sr = sf.read(str(file_path), dtype='float32')
        feats = extract_lfcc_features(y, sr=sr)  # (T, 60)
        all_frames.append(feats)
        
        if sum(len(f) for f in all_frames) >= max_frames:
            break
            
    stacked = np.vstack(all_frames)
    if len(stacked) > max_frames:
        idx = np.random.choice(len(stacked), size=max_frames, replace=False)
        stacked = stacked[idx]
    print(f"Total frame {target_label}: {stacked.shape}")
    return stacked

# Kumpulkan frame bonafide dan spoof (misal 300.000 frame per kelas untuk efisiensi GMM)
MAX_FRAMES_PER_CLASS = 300000
frames_bonafide = collect_training_frames(train_df, train_audio_dir, 'bonafide', max_frames=MAX_FRAMES_PER_CLASS)
frames_spoof = collect_training_frames(train_df, train_audio_dir, 'spoof', max_frames=MAX_FRAMES_PER_CLASS)

## 6. Pelatihan Model GMM (B0: LFCC-GMM)
Sesuai baseline resmi ASVspoof 2019:
- 512 komponen Gaussian (atau 256/128 untuk iterasi cepat)
- Covariance type: `diag`

In [ ]:
N_COMPONENTS = 256  # 256 atau 512 components

print(f"\nMemulai training GMM Bonafide ({N_COMPONENTS} components)... [harap tunggu]")
gmm_bonafide = GaussianMixture(
    n_components=N_COMPONENTS,
    covariance_type='diag',
    max_iter=100,
    random_state=42,
    verbose=1
)
gmm_bonafide.fit(frames_bonafide)
print("✅ GMM Bonafide selesai!")

print(f"\nMemulai training GMM Spoof ({N_COMPONENTS} components)... [harap tunggu]")
gmm_spoof = GaussianMixture(
    n_components=N_COMPONENTS,
    covariance_type='diag',
    max_iter=100,
    random_state=42,
    verbose=1
)
gmm_spoof.fit(frames_spoof)
print("✅ GMM Spoof selesai!")

## 7. Evaluasi pada Development Set & Perhitungan EER Resmi
Skor per utterance $X$:
$$\text{score}(X) = \frac{1}{T} \sum_{t=1}^T [ \log P(x_t \mid \mathcal{M}_{\text{bonafide}}) - \log P(x_t \mid \mathcal{M}_{\text{spoof}}) ]$$
Skor lebih tinggi $\rightarrow$ bona fide; skor lebih rendah $\rightarrow$ spoof.

In [ ]:
def compute_eer(bonafide_scores, spoof_scores):
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    scores = np.concatenate([bonafide_scores, spoof_scores])
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return eer, thresholds[idx]

print("Menghitung skor evaluasi untuk Development Set...")
dev_scores = []
dev_labels = []

for _, row in tqdm(dev_df.iterrows(), total=len(dev_df)):
    file_path = dev_audio_dir / f"{row['audio_id']}.flac"
    if not file_path.exists():
        continue
    y, sr = sf.read(str(file_path), dtype='float32')
    feats = extract_lfcc_features(y, sr=sr)  # (T, 60)
    
    llk_bonafide = gmm_bonafide.score_samples(feats)
    llk_spoof = gmm_spoof.score_samples(feats)
    score = np.mean(llk_bonafide - llk_spoof)
    
    dev_scores.append(score)
    dev_labels.append(row['label'])

dev_scores = np.array(dev_scores)
dev_labels = np.array(dev_labels)

bona_scores = dev_scores[dev_labels == 'bonafide']
spoof_scores = dev_scores[dev_labels == 'spoof']

eer_dev, threshold = compute_eer(bona_scores, spoof_scores)

print("=" * 60)
print(f"🎯 HASIL EVALUASI BASELINE B0 (LFCC-GMM):")
print(f"   - EER Dev:       {eer_dev * 100:.2f}%")
print(f"   - Threshold EER: {threshold:.4f}")
print(f"   - Bona fide mean score: {bona_scores.mean():.4f}")
print(f"   - Spoof mean score:     {spoof_scores.mean():.4f}")
print("=" * 60)

## 8. Visualisasi Distribusi Skor dan Kurva Evaluasi

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(bona_scores, bins=50, alpha=0.6, label='Bona fide', color='green', density=True)
plt.hist(spoof_scores, bins=50, alpha=0.6, label='Spoof', color='red', density=True)
plt.axvline(threshold, color='black', linestyle='--', label=f'EER Threshold ({threshold:.3f})')
plt.title(f'Distribusi Skor Log-Likelihood Ratio B0 (LFCC-GMM) — EER: {eer_dev * 100:.2f}%')
plt.xlabel('Log-Likelihood Ratio Score')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('b0_score_distribution.png', dpi=300)
plt.show()

## 9. Simpan Model dan Log Hasil
Simpan artefak model untuk laporan dan reproducibilitas.

In [ ]:
model_path = Path('models') / 'b0_lfcc_gmm.pkl'
model_path.parent.mkdir(parents=True, exist_ok=True)

with open(model_path, 'wb') as f:
    pickle.dump({'gmm_bonafide': gmm_bonafide, 'gmm_spoof': gmm_spoof, 'eer_dev': eer_dev, 'threshold': threshold}, f)

log_df = pd.DataFrame([{
    'Model': 'B0 LFCC-GMM',
    'Fitur': 'LFCC (20 static + delta + delta2 = 60-dim)',
    'Window': '4 s / full',
    'Trimming': 'tidak',
    'Seed': 42,
    'Components': N_COMPONENTS,
    'EER dev': f'{eer_dev * 100:.2f}%'
}])

log_df.to_csv('b0_evaluation_log.csv', index=False)
print("✅ Log disimpan ke b0_evaluation_log.csv")
display(log_df)